In [5]:
from arraylake import Client, config
import xbatcher

from main import XBatcherPyTorchDataset
from torch.utils.data import DataLoader


In [ ]:
client = Client()

repo = client.get_repo("earthmover-public/weatherbench2")

# opened with xarray/dask
ds = repo.to_xarray("datasets/era5/1959-2022-6h-128x64_equiangular_with_poles_conservative", chunks={})
ds

**Pytorch DataLoader via Xbatcher**

We'll use Xarray, Xbatcher and PyTorch to create a dataloader we can use to train a simple auto encoder.

In [21]:
DEFAULT_VARS = [
    "10m_wind_speed",
    "2m_temperature",
    "specific_humidity",
]
patch_size = 32
input_steps = 3
output_steps = 0
jitter = 16

ds = ds[DEFAULT_VARS]
patch = dict(
    latitude=patch_size + jitter,
    longitude=patch_size + jitter,
    time=input_steps + output_steps,
)
overlap = dict(latitude=32, longitude=32, time=input_steps // 3 * 2)

# Xbatcher BatchGenerator
bgen = xbatcher.BatchGenerator(
    ds,
    input_dims=patch,
    input_overlap=overlap,
    preload_batch=False,
)

# Pytorch Dataset
dataset = XBatcherPyTorchDataset(bgen)

# Pytorch DataLoader
data_params = {
    "batch_size": 16,
    "shuffle": True
}

dataloader = DataLoader(dataset, **data_params)

In [22]:
dataset[0].shape

torch.Size([3, 15, 48, 48])

**Train a simple autoencoder**

*Note: this model is not meant to be scientifically meaningful. Instead it is meant to illustrate that data from Arraylake can be easily wired all the way through to a real model.*

In [29]:
from model import Autoencoder

# Initialize the model, loss function, and optimizer
model = Autoencoder()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
batches_per_epoch = 10

# Training function
def train_autoencoder(dataloader, model, criterion, optimizer, epochs=20):
    for epoch in range(epochs):
        for i, batch in enumerate(dataloader):
            batch = batch.view(-1, 15, 48, 48)  # Flatten batch_size * 3
            # Forward pass
            outputs = model(batch)
            loss = criterion(outputs, batch)
            
            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            if i+1 > batches_per_epoch:
                break
        
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

In [30]:
train_autoencoder(dataloader, model, criterion, optimizer)

torch.Size([16, 3, 15, 48, 48])
torch.Size([48, 15, 48, 48])
torch.Size([16, 3, 15, 48, 48])
torch.Size([48, 15, 48, 48])
torch.Size([16, 3, 15, 48, 48])
torch.Size([48, 15, 48, 48])
torch.Size([16, 3, 15, 48, 48])
torch.Size([48, 15, 48, 48])
torch.Size([16, 3, 15, 48, 48])
torch.Size([48, 15, 48, 48])
torch.Size([16, 3, 15, 48, 48])
torch.Size([48, 15, 48, 48])
torch.Size([16, 3, 15, 48, 48])
torch.Size([48, 15, 48, 48])
torch.Size([16, 3, 15, 48, 48])
torch.Size([48, 15, 48, 48])
torch.Size([16, 3, 15, 48, 48])
torch.Size([48, 15, 48, 48])
torch.Size([16, 3, 15, 48, 48])
torch.Size([48, 15, 48, 48])
torch.Size([16, 3, 15, 48, 48])
torch.Size([48, 15, 48, 48])
Epoch [1/20], Loss: 5372.1494
torch.Size([16, 3, 15, 48, 48])
torch.Size([48, 15, 48, 48])
torch.Size([16, 3, 15, 48, 48])
torch.Size([48, 15, 48, 48])
torch.Size([16, 3, 15, 48, 48])
torch.Size([48, 15, 48, 48])
torch.Size([16, 3, 15, 48, 48])
torch.Size([48, 15, 48, 48])
torch.Size([16, 3, 15, 48, 48])
torch.Size([48, 15, 48,

KeyboardInterrupt: 